# Goofy Screener — Phase 5: Position Sizing
### Kelly Criterion + Volatility Scaling

**Builds on:** Phase 4 (regime-aware verdict)  
**New question:** You know *which* asset and *whether* to trade today — but **how much**?

---

## Two methods, combined:

| Method | What it answers | Based on |
|--------|----------------|----------|
| **Half-Kelly** | How confident are we in this edge? | Win rate & avg win/loss |
| **Vol Scaling** | How wild is this asset right now? | Recent 21-day realized vol |

**Final size = Half-Kelly × Vol Scalar, capped at 100%**

---

## Kelly Formula (simplified)

```
Full Kelly:  f* = p - q/b
  where  p = win probability
         q = 1 - p
         b = avg_win / avg_loss

Half Kelly:  f = f* × 0.5
```

We always use **half-Kelly** — full Kelly is mathematically optimal but leads to brutal drawdowns in practice.

In [1]:
# ── Imports ───────────────────────────────────────────────────────────────────
import yfinance as yf
import numpy as np
import pandas as pd
import datetime as dt
import os, warnings
warnings.filterwarnings("ignore")

from regime_detector import label_regimes, get_allowed_regimes, load_asset_gates, ASSET_SPECIFIC_GATES
from position_sizer import compute_trade_stats, recommend_size, half_kelly, vol_scalar

print("Imports OK")

Imports OK


---
## Part 1 — Understanding Kelly Criterion
Before running the full screener, let's build intuition for the Kelly formula.

In [2]:
# ── Kelly intuition: how does win rate + win/loss ratio affect position size? ──
print("Half-Kelly position sizes for different strategy profiles:\n")
print(f"{'Win Rate':>10} {'Avg Win':>9} {'Avg Loss':>9} {'Win/Loss':>9} {'Half-Kelly':>11}")
print("-" * 52)

cases = [
    (0.55, 0.10, 0.05),   # 55% WR, 2:1 ratio — typical trend-following
    (0.60, 0.08, 0.06),   # 60% WR, 1.33:1 — high WR, lower ratio
    (0.45, 0.15, 0.05),   # 45% WR, 3:1   — low WR, big winners
    (0.40, 0.05, 0.08),   # 40% WR, 0.6:1 — negative edge (skip)
    (0.50, 0.12, 0.08),   # 50% WR, 1.5:1 — coin flip with bigger wins
]

for wr, aw, al in cases:
    k = half_kelly(wr, aw, al)
    ratio = aw / al
    print(f"  {wr*100:>7.0f}%   {aw*100:>7.1f}%   {al*100:>7.1f}%   {ratio:>7.2f}x   {k*100:>9.1f}%")

Half-Kelly position sizes for different strategy profiles:

  Win Rate   Avg Win  Avg Loss  Win/Loss  Half-Kelly
----------------------------------------------------
       55%      10.0%       5.0%      2.00x        16.2%
       60%       8.0%       6.0%      1.33x        15.0%
       45%      15.0%       5.0%      3.00x        13.3%
       40%       5.0%       8.0%      0.62x         0.0%
       50%      12.0%       8.0%      1.50x         8.3%


In [3]:
# ── Vol scaling intuition: calm vs wild assets ────────────────────────────────
TARGET_VOL = 0.15  # 15% annual vol target

np.random.seed(42)
print(f"Vol Scaling — target: {TARGET_VOL*100:.0f}% annual vol\n")
print(f"{'Asset Type':>22} {'Simulated Ann Vol':>18} {'Vol Scalar':>11} {'Effect':>20}")
print("-" * 75)

scenarios = [
    ("ASX ETF (calm)",  0.10),
    ("US blue-chip",    0.15),
    ("JP bank stock",   0.22),
    ("Energy stock",    0.30),
    ("TSLA/AMD (wild)", 0.50),
]

for label, ann_vol in scenarios:
    daily_vol = ann_vol / (252**0.5)
    fake_returns = pd.Series(np.random.normal(0, daily_vol, 30))
    scalar = vol_scalar(fake_returns, target_vol=TARGET_VOL)
    effect = "scale UP" if scalar > 1 else "scale DOWN"
    print(f"  {label:>20}   {ann_vol*100:>14.0f}%   {scalar:>10.2f}x   {effect:>20}")

Vol Scaling — target: 15% annual vol

            Asset Type  Simulated Ann Vol  Vol Scalar               Effect
---------------------------------------------------------------------------
        ASX ETF (calm)               10%         1.81x               scale UP
          US blue-chip               15%         1.25x               scale UP
         JP bank stock               22%         0.65x             scale DOWN
          Energy stock               30%         0.50x             scale DOWN
       TSLA/AMD (wild)               50%         0.32x             scale DOWN


---
## Part 2 — Config & Universes

In [4]:
# ── Config ────────────────────────────────────────────────────────────────────
TRAIN_START      = dt.datetime(2016, 1, 1)
TRAIN_END        = dt.datetime(2021, 1, 1)
TEST_END         = dt.datetime.now()
MIN_ROWS         = 400
PERIODS_PER_YEAR = 252
TARGET_VOL       = 0.15   # 15% annualised vol target
KELLY_FRACTION   = 0.5    # half-Kelly
VOL_LOOKBACK     = 21     # ~1 month

TIER_S = {"sharpe": 0.8,  "ret": 30,  "max_dd": -20}
TIER_A = {"sharpe": 0.4,  "ret": 10,  "max_dd": -35}
TIER_B = {"sharpe": 0.1,  "ret": -10, "max_dd": -50}

SCRIPT_DIR = os.path.abspath(".")
OUTPUT_DIR = os.path.join(SCRIPT_DIR, "screener_output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

LONG_TO_SHORT = {
    "MA Crossover": "MA", "RSI": "RSI", "Bollinger Bands": "BB",
    "MACD": "MACD", "Mean Reversion": "MeanReversion",
}

print("Config set.")

Config set.


In [5]:
# ── Stock universes ───────────────────────────────────────────────────────────
US_UNIVERSE = [
    "AAPL","MSFT","NVDA","GOOGL","META","AMZN","TSLA","AMD",
    "JPM","BAC","GS","MS","V","MA","BRK-B",
    "JNJ","UNH","LLY","PFE","ABBV","MRK",
    "XOM","CVX","COP",
    "PG","KO","PEP","WMT","COST","MCD",
    "CAT","DE","BA","HON","RTX",
    "SPY","QQQ","GLD","TLT","IWM",
]

ASX_UNIVERSE = [
    "CBA.AX","WBC.AX","ANZ.AX","NAB.AX",
    "BHP.AX","RIO.AX","FMG.AX","S32.AX",
    "WDS.AX","STO.AX","BPT.AX",
    "CSL.AX","RMD.AX","COH.AX","SHL.AX",
    "WES.AX","WOW.AX","COL.AX","JBH.AX",
    "XRO.AX","WTC.AX","ALU.AX",
    "TCL.AX","APA.AX","SKI.AX",
    "GMG.AX","SCG.AX","GPT.AX",
    "IOZ.AX","STW.AX","VAS.AX",
]

JPX_UNIVERSE = [
    "7203.T","7267.T","7261.T","7272.T","7269.T",
    "6758.T","6501.T","6954.T","6902.T","6861.T",
    "6762.T","8035.T","6857.T","6723.T","6594.T",
    "4063.T","4523.T",
    "9432.T","9433.T","9434.T",
    "9984.T","4689.T","6098.T","4385.T",
    "7974.T","9684.T","7832.T",
    "8306.T","8316.T","8411.T",
    "8750.T","8725.T",
    "3382.T","8267.T","4661.T","2914.T",
    "6301.T","6326.T","7011.T",
    "4502.T","4519.T",
    "8001.T","8002.T","8058.T",
    "9022.T","9020.T",
    "1321.T","1306.T",
]

UNIVERSE_MAP = {"US": US_UNIVERSE, "ASX": ASX_UNIVERSE, "JPX": JPX_UNIVERSE}
print(f"Universes: US={len(US_UNIVERSE)}, ASX={len(ASX_UNIVERSE)}, JPX={len(JPX_UNIVERSE)}")

Universes: US=40, ASX=31, JPX=48


---
## Part 3 — Strategy Functions & Metrics

In [6]:
# ── Metrics calculator ─────────────────────────────────────────────────────────
def compute_metrics(price, position):
    df = pd.DataFrame({"price": price})
    df["pos"]       = position.reindex(df.index).fillna(0)
    df["log_ret"]   = np.log(df["price"] / df["price"].shift(1))
    df["strat_ret"] = df["pos"] * df["log_ret"]
    df["cum"]       = np.exp(df["strat_ret"].cumsum())
    lr = df["strat_ret"].dropna()
    if len(lr) < 10:
        return {"Sharpe": np.nan, "TotalRet": np.nan, "MaxDD": np.nan, "WinRate": np.nan}
    ann    = np.exp(lr.mean() * PERIODS_PER_YEAR) - 1
    vol    = lr.std() * np.sqrt(PERIODS_PER_YEAR)
    sharpe = ann / vol if vol != 0 else np.nan
    cum    = df["cum"].dropna()
    mdd    = ((cum - cum.cummax()) / cum.cummax()).min() if len(cum) > 0 else np.nan
    total  = cum.iloc[-1] - 1 if len(cum) > 0 else np.nan
    wins   = (lr > 0).sum() / max(len(lr), 1)
    return {
        "Sharpe":   round(float(sharpe), 3) if not np.isnan(sharpe) else np.nan,
        "TotalRet": round(float(total) * 100, 2) if not np.isnan(total) else np.nan,
        "MaxDD":    round(float(mdd) * 100, 2) if not np.isnan(mdd) else np.nan,
        "WinRate":  round(float(wins) * 100, 1) if not np.isnan(wins) else np.nan,
    }

print("compute_metrics() ready")

compute_metrics() ready


In [7]:
# ── Strategy functions ────────────────────────────────────────────────────────
def strategy_ma(p, short=20, long=50):
    return (p.rolling(short).mean() > p.rolling(long).mean()).astype(int).shift(1)

def strategy_rsi(p, period=14, oversold=30, overbought=70):
    d = p.diff()
    rsi = 100 - (100 / (1 + d.clip(lower=0).rolling(period).mean() /
                            (-d.clip(upper=0)).rolling(period).mean()))
    sig = np.zeros(len(p)); hold = False
    for i, r in enumerate(rsi):
        if np.isnan(r): sig[i] = 0
        elif not hold and r < oversold:  hold = True;  sig[i] = 1
        elif hold and r > overbought:    hold = False; sig[i] = 0
        else:                            sig[i] = 1 if hold else 0
    return pd.Series(sig, index=p.index).shift(1)

def strategy_bb(p, window=20, num_std=2.0):
    mid = p.rolling(window).mean()
    low = mid - num_std * p.rolling(window).std()
    sig = np.zeros(len(p)); hold = False
    for i in range(len(p)):
        pi, m, l = p.iloc[i], mid.iloc[i], low.iloc[i]
        if np.isnan(l): sig[i] = 0
        elif not hold and pi <= l: hold = True;  sig[i] = 1
        elif hold and pi >= m:     hold = False; sig[i] = 0
        else:                      sig[i] = 1 if hold else 0
    return pd.Series(sig, index=p.index).shift(1)

def strategy_macd(p, fast=12, slow=26, signal_p=9):
    macd = p.ewm(span=fast).mean() - p.ewm(span=slow).mean()
    return (macd > macd.ewm(span=signal_p).mean()).astype(int).shift(1)

def strategy_mr(p, window=20, threshold=1.5):
    z = (p - p.rolling(window).mean()) / p.rolling(window).std()
    sig = np.zeros(len(p)); hold = False
    for i, zi in enumerate(z):
        if np.isnan(zi): sig[i] = 0
        elif not hold and zi < -threshold: hold = True;  sig[i] = 1
        elif hold and zi >= 0:             hold = False; sig[i] = 0
        else:                              sig[i] = 1 if hold else 0
    return pd.Series(sig, index=p.index).shift(1)

STRATEGY_FNS = {
    "MA Crossover": strategy_ma, "RSI": strategy_rsi,
    "Bollinger Bands": strategy_bb, "MACD": strategy_macd,
    "Mean Reversion": strategy_mr,
}

STRATEGY_GRIDS = {
    "MA Crossover":    [{"short": s, "long": l} for s in [10,20,50] for l in [50,100,200] if s < l],
    "RSI":             [{"period": p, "oversold": os, "overbought": ob} for p in [10,14,21] for os, ob in [(25,65),(30,70),(35,75)]],
    "Bollinger Bands": [{"window": w, "num_std": s} for w in [10,20,30] for s in [1.5,2.0,2.5]],
    "MACD":            [{"fast": f, "slow": s, "signal_p": sg} for f in [8,12] for s in [21,26] for sg in [7,9] if f < s],
    "Mean Reversion":  [{"window": w, "threshold": t} for w in [10,20,40] for t in [1.0,1.5,2.0]],
}

print(f"5 strategies ready — total param combos: {sum(len(v) for v in STRATEGY_GRIDS.values())}")

5 strategies ready — total param combos: 43


---
## Part 4 — Scoring, Regime Verdict & Position Sizing

In [8]:
# ── Scoring & tiering ────────────────────────────────────────────────────────
def score_asset(row):
    sharpe  = row.get("OUT Sharpe", np.nan)
    ret     = row.get("OUT Strat Ret %", np.nan)
    mdd     = row.get("OUT Strat Max DD %", np.nan)
    dd_save = row.get("DD Saved %", np.nan)
    if any(pd.isna(v) for v in [sharpe, ret, mdd]):
        return ("Skip", 0.0)
    s_pts  = min(max(sharpe / 1.5, 0), 1) * 40
    r_pts  = min(max(ret + 30, 0) / 130, 1) * 25
    d_pts  = min(max((60 + mdd) / 60, 0), 1) * 20
    ds_pts = min(max((dd_save + 20) / 60, 0), 1) * 15 if not pd.isna(dd_save) else 0
    score  = round(s_pts + r_pts + d_pts + ds_pts, 1)
    if sharpe >= TIER_S["sharpe"] and ret >= TIER_S["ret"] and mdd >= TIER_S["max_dd"]:   tier = "S"
    elif sharpe >= TIER_A["sharpe"] and ret >= TIER_A["ret"] and mdd >= TIER_A["max_dd"]: tier = "A"
    elif sharpe >= TIER_B["sharpe"] and mdd >= TIER_B["max_dd"]:                          tier = "B"
    else:                                                                                   tier = "Skip"
    return (tier, score)

# ── Phase 4 regime verdict ────────────────────────────────────────────────────
def compute_today_verdict(asset, best_strategy_long, ohlc):
    blank = {"Current Trend": "—", "Current Vol": "—", "Allowed Regimes": "—", "Today's Verdict": "—"}
    if ohlc is None or ohlc.empty: return blank
    if not {"High", "Low", "Close"}.issubset(set(ohlc.columns)): return blank
    try:
        labelled  = label_regimes(ohlc)
        trend_now = labelled["Trend"].dropna()
        vol_now   = labelled["Vol"].dropna()
        if trend_now.empty: return blank
        cur_trend = trend_now.iloc[-1]
        cur_vol   = vol_now.iloc[-1] if not vol_now.empty else "—"
    except: return blank
    short = LONG_TO_SHORT.get(best_strategy_long)
    if short is None:
        return {"Current Trend": cur_trend, "Current Vol": cur_vol, "Allowed Regimes": "—", "Today's Verdict": "—"}
    allowed = get_allowed_regimes(asset, short)
    verdict = "TRADE" if (not allowed or cur_trend in allowed) else "STAND DOWN"
    return {
        "Current Trend":   cur_trend,
        "Current Vol":     cur_vol,
        "Allowed Regimes": ", ".join(sorted(allowed)) if allowed else "(no gate)",
        "Today's Verdict": verdict,
    }

print("Scoring, regime verdict, and sizing helpers ready")

Scoring, regime verdict, and sizing helpers ready


---
## Part 5 — Choose Your Market & Download Data
Change `MARKET` to `"US"`, `"ASX"`, `"JPX"`, or `"ALL"`

In [9]:
# ── Choose market ─────────────────────────────────────────────────────────────
MARKET = "ALL"   # ← change to "US", "ASX", or "JPX" for a faster run

markets_to_run = ["US", "ASX", "JPX"] if MARKET == "ALL" else [MARKET]

all_assets = []
for m in markets_to_run:
    all_assets.extend(UNIVERSE_MAP[m])
seen = set(); unique_assets = []
for a in all_assets:
    if a not in seen: unique_assets.append(a); seen.add(a)

# Load empirical asset-specific gates from Phase 4b
loaded_gates = load_asset_gates()
if loaded_gates:
    print(f"Phase 4 gates loaded: {sum(len(v) for v in loaded_gates.values())} gates across {len(loaded_gates)} assets")
else:
    print("No gates file found — using theory defaults")

print(f"\nMarkets: {markets_to_run}")
print(f"Assets to download: {len(unique_assets)}")

Phase 4 gates loaded: 4 gates across 3 assets

Markets: ['US', 'ASX', 'JPX']
Assets to download: 119


In [10]:
# ── Download price data ───────────────────────────────────────────────────────
# This cell takes ~2-3 minutes for ALL markets
price_data = {}   # Close series — for backtest
ohlc_data  = {}   # Full OHLC   — for regime verdict

for asset in unique_assets:
    try:
        raw = yf.download(asset, start=TRAIN_START, end=TEST_END,
                          auto_adjust=True, progress=False)
        if not raw.empty and len(raw) >= MIN_ROWS:
            if isinstance(raw.columns, pd.MultiIndex):
                raw.columns = raw.columns.get_level_values(0)
            close = raw["Close"].squeeze()
            if isinstance(close, pd.DataFrame):
                close = close.iloc[:, 0]
            price_data[asset] = close
            if {"High", "Low", "Close"}.issubset(set(raw.columns)):
                ohlc_data[asset] = raw[["Open", "High", "Low", "Close"]].copy() \
                    if "Open" in raw.columns else raw[["High", "Low", "Close"]].copy()
            print(f"  ✓ {asset:14} | {len(raw)} rows")
        else:
            print(f"  ✗ {asset:14} | skipped ({len(raw)} rows)")
    except Exception as e:
        print(f"  ✗ {asset:14} | error: {e}")

print(f"\n→ {len(price_data)} assets ready ({len(ohlc_data)} with OHLC)")

  ✓ AAPL           | 2602 rows
  ✓ MSFT           | 2602 rows
  ✓ NVDA           | 2602 rows
  ✓ GOOGL          | 2602 rows
  ✓ META           | 2602 rows
  ✓ AMZN           | 2602 rows
  ✓ TSLA           | 2602 rows
  ✓ AMD            | 2602 rows
  ✓ JPM            | 2602 rows
  ✓ BAC            | 2602 rows
  ✓ GS             | 2602 rows
  ✓ MS             | 2602 rows
  ✓ V              | 2602 rows
  ✓ MA             | 2602 rows
  ✓ BRK-B          | 2602 rows
  ✓ JNJ            | 2602 rows
  ✓ UNH            | 2602 rows
  ✓ LLY            | 2602 rows
  ✓ PFE            | 2602 rows
  ✓ ABBV           | 2602 rows
  ✓ MRK            | 2602 rows
  ✓ XOM            | 2602 rows
  ✓ CVX            | 2602 rows
  ✓ COP            | 2602 rows
  ✓ PG             | 2602 rows
  ✓ KO             | 2602 rows
  ✓ PEP            | 2602 rows
  ✓ WMT            | 2602 rows
  ✓ COST           | 2602 rows
  ✓ MCD            | 2602 rows
  ✓ CAT            | 2602 rows
  ✓ DE             | 2602 rows
  ✓ BA  

$ALU.AX: possibly delisted; no timezone found

1 Failed download:
['ALU.AX']: possibly delisted; no timezone found


  ✗ ALU.AX         | skipped (0 rows)
  ✓ TCL.AX         | 2619 rows
  ✓ APA.AX         | 2619 rows


$SKI.AX: possibly delisted; no timezone found

1 Failed download:
['SKI.AX']: possibly delisted; no timezone found


  ✗ SKI.AX         | skipped (0 rows)
  ✓ GMG.AX         | 2619 rows
  ✓ SCG.AX         | 2619 rows
  ✓ GPT.AX         | 2619 rows
  ✓ IOZ.AX         | 2618 rows
  ✓ STW.AX         | 2618 rows
  ✓ VAS.AX         | 2618 rows
  ✓ 7203.T         | 2548 rows
  ✓ 7267.T         | 2548 rows
  ✓ 7261.T         | 2548 rows
  ✓ 7272.T         | 2548 rows
  ✓ 7269.T         | 2548 rows
  ✓ 6758.T         | 2548 rows
  ✓ 6501.T         | 2548 rows
  ✓ 6954.T         | 2548 rows
  ✓ 6902.T         | 2548 rows
  ✓ 6861.T         | 2548 rows
  ✓ 6762.T         | 2548 rows
  ✓ 8035.T         | 2548 rows
  ✓ 6857.T         | 2548 rows
  ✓ 6723.T         | 2548 rows
  ✓ 6594.T         | 2548 rows
  ✓ 4063.T         | 2548 rows
  ✓ 4523.T         | 2548 rows
  ✓ 9432.T         | 2548 rows
  ✓ 9433.T         | 2548 rows
  ✓ 9434.T         | 1798 rows
  ✓ 9984.T         | 2548 rows
  ✓ 4689.T         | 2548 rows
  ✓ 6098.T         | 2548 rows
  ✓ 4385.T         | 1929 rows
  ✓ 7974.T         | 2548 rows
 

---
## Part 6 — Run the Screener
This is the main loop — strategy search + regime verdict + position sizing for every asset.

In [11]:
# ── Main screening loop ───────────────────────────────────────────────────────
# Takes 5-10 minutes for ALL markets
today   = dt.datetime.now().strftime("%Y-%m-%d")
results = []

for market_name in markets_to_run:
    assets = UNIVERSE_MAP[market_name]
    valid  = [a for a in assets if a in price_data]
    print(f"\n── {market_name} ({len(valid)} assets) ──")

    for asset in assets:
        if asset not in price_data: continue
        full  = price_data[asset]
        train = full[full.index < TRAIN_END]
        test  = full[full.index >= TRAIN_END]
        if len(train) < 100 or len(test) < 50: continue

        # Best strategy on train period
        best_sharpe, best_strat, best_params = -999, None, None
        for strat_name, fn in STRATEGY_FNS.items():
            for params in STRATEGY_GRIDS[strat_name]:
                try:
                    pos = fn(train, **params)
                    m   = compute_metrics(train, pos)
                    if not np.isnan(m["Sharpe"]) and m["Sharpe"] > best_sharpe:
                        best_sharpe, best_strat, best_params = m["Sharpe"], strat_name, params
                except: continue
        if best_strat is None: continue

        # Out-of-sample
        pos_test = STRATEGY_FNS[best_strat](test, **best_params)
        tm       = compute_metrics(test, pos_test)

        # B&H benchmark
        log     = np.log(test / test.shift(1)).dropna()
        bah_ret = (np.exp(log.cumsum()).iloc[-1] - 1) * 100 if len(log) > 0 else np.nan
        bah_cum = np.exp(log.cumsum()) if len(log) > 0 else pd.Series(dtype=float)
        bah_mdd = ((bah_cum - bah_cum.cummax()) / bah_cum.cummax()).min() * 100 if len(bah_cum) > 0 else np.nan
        dd_saved = round(tm["MaxDD"] - float(bah_mdd), 2) if not np.isnan(bah_mdd) else np.nan

        # Phase 4 regime verdict
        verdict = compute_today_verdict(asset, best_strat, ohlc_data.get(asset))

        # Phase 5 position sizing
        trade_stats    = compute_trade_stats(test, pos_test)
        recent_returns = test.pct_change().dropna()
        sizing         = recommend_size(trade_stats, recent_returns,
                                        target_vol=TARGET_VOL, kelly_fraction=KELLY_FRACTION)

        row = {
            "Market": market_name, "Asset": asset,
            "Best Strategy": best_strat, "Best Params": str(best_params),
            "Train Sharpe": round(best_sharpe, 3),
            "OUT Sharpe": tm["Sharpe"], "OUT Win Rate %": tm["WinRate"],
            "OUT Strat Ret %": tm["TotalRet"], "OUT B&H Ret %": round(float(bah_ret), 2) if not np.isnan(bah_ret) else np.nan,
            "OUT Strat Max DD %": tm["MaxDD"], "OUT B&H Max DD %": round(float(bah_mdd), 2) if not np.isnan(bah_mdd) else np.nan,
            "DD Saved %": dd_saved, "Beats B&H": (tm["TotalRet"] or 0) > (bah_ret or 0),
            "Current Trend": verdict["Current Trend"], "Current Vol": verdict["Current Vol"],
            "Allowed Regimes": verdict["Allowed Regimes"], "Today's Verdict": verdict["Today's Verdict"],
            "N Trades": trade_stats["n_trades"] if trade_stats else None,
            "Trade Win Rate %": round(trade_stats["win_rate"] * 100, 1) if trade_stats else None,
            "Avg Win %": round(trade_stats["avg_win"] * 100, 2) if trade_stats else None,
            "Avg Loss %": round(trade_stats["avg_loss"] * 100, 2) if trade_stats else None,
            "Kelly %": sizing["Kelly %"], "Vol Scalar": sizing["Vol Scalar"],
            "Recommended Size %": sizing["Recommended Size %"],
            "Run Date": today,
        }
        tier, score = score_asset(row)
        row["Tier"] = tier; row["Score"] = score
        results.append(row)

        tier_icon = {"S": "⭐", "A": "✅", "B": "🔵", "Skip": "⬜"}.get(tier, "")
        v = verdict["Today's Verdict"]
        v_icon = {"TRADE": "🟢", "STAND DOWN": "🔴"}.get(v, "  ")
        sz = f"{sizing['Recommended Size %']:.0f}%" if sizing["Recommended Size %"] is not None else "—"
        print(f"  {tier_icon} [{tier:4}] {asset:14} → {best_strat:18} | "
              f"Sharpe:{tm['Sharpe']:5.2f} | {v_icon} {verdict['Current Trend']:8} → {v:10} | Size:{sz}")

df_results = pd.DataFrame(results)
print(f"\nDone — {len(df_results)} assets processed")


── US (40 assets) ──
  ✅ [A   ] AAPL           → MACD               | Sharpe: 0.69 | 🟢 Bull     → TRADE      | Size:6%
  🔵 [B   ] MSFT           → MA Crossover       | Sharpe: 0.37 | 🔴 Bear     → STAND DOWN | Size:7%
  ⬜ [Skip] NVDA           → MA Crossover       | Sharpe: 0.97 | 🟢 Bull     → TRADE      | Size:11%
  🔵 [B   ] GOOGL          → RSI                | Sharpe: 0.16 | 🔴 Bull     → STAND DOWN | Size:8%
  ⬜ [Skip] META           → RSI                | Sharpe:-0.10 | 🟢 Sideways → TRADE      | Size:2%
  🔵 [B   ] AMZN           → MA Crossover       | Sharpe: 0.28 | 🟢 Bull     → TRADE      | Size:4%
  🔵 [B   ] TSLA           → MACD               | Sharpe: 0.89 | 🟢 Bull     → TRADE      | Size:7%
  ⬜ [Skip] AMD            → MA Crossover       | Sharpe: 0.50 | 🟢 Bull     → TRADE      | Size:1%
  ✅ [A   ] JPM            → MA Crossover       | Sharpe: 0.68 | 🔴 Sideways → STAND DOWN | Size:19%
  🔵 [B   ] BAC            → MACD               | Sharpe: 0.93 | 🟢 Bull     → TRADE      | Size

---
## Part 7 — Summary & Analysis

In [12]:
# ── Tier & verdict summary ────────────────────────────────────────────────────
print(f"PHASE 5 SUMMARY — {today}")
print(f"{'='*60}")
print(f"Total assets: {len(df_results)}\n")

print("Tier Breakdown:")
for tier in ["S", "A", "B", "Skip"]:
    icon  = {"S": "⭐", "A": "✅", "B": "🔵", "Skip": "⬜"}[tier]
    count = len(df_results[df_results["Tier"] == tier])
    print(f"  {icon} {tier:5}: {count:3}  {'█' * count}")

print("\nToday's Verdict:")
for v in ["TRADE", "STAND DOWN", "—"]:
    icon  = {"TRADE": "🟢", "STAND DOWN": "🔴", "—": "⬜"}[v]
    count = len(df_results[df_results["Today's Verdict"] == v])
    print(f"  {icon} {v:11}: {count:3}  {'█' * count}")

PHASE 5 SUMMARY — 2026-05-11
Total assets: 117

Tier Breakdown:
  ⭐ S    :   8  ████████
  ✅ A    :  36  ████████████████████████████████████
  🔵 B    :  33  █████████████████████████████████
  ⬜ Skip :  40  ████████████████████████████████████████

Today's Verdict:
  🟢 TRADE      :  45  █████████████████████████████████████████████
  🔴 STAND DOWN :  72  ████████████████████████████████████████████████████████████████████████
  ⬜ —          :   0  


In [13]:
# ── S-Tier assets ─────────────────────────────────────────────────────────────
s_tier = df_results[df_results["Tier"] == "S"].sort_values("Score", ascending=False)
cols = ["Market", "Asset", "Best Strategy", "Score", "OUT Sharpe",
        "OUT Strat Ret %", "OUT Strat Max DD %", "Today's Verdict",
        "Kelly %", "Vol Scalar", "Recommended Size %"]
print("S-Tier Assets (Excellent):\n")
print(s_tier[cols].to_string(index=False))

S-Tier Assets (Excellent):

Market  Asset   Best Strategy  Score  OUT Sharpe  OUT Strat Ret %  OUT Strat Max DD % Today's Verdict  Kelly %  Vol Scalar  Recommended Size %
   JPX 8725.T             RSI   87.6       1.296           110.06               -8.81      STAND DOWN      NaN         NaN                 NaN
   JPX 8750.T            MACD   75.8       1.065           186.14              -16.87           TRADE     14.4      0.5180                 7.5
   JPX 8411.T            MACD   74.7       0.999           145.64              -17.74           TRADE     13.1      0.3715                 4.9
   JPX 8306.T            MACD   74.5       1.053           161.29              -19.71           TRADE     16.4      0.5693                 9.3
   ASX CBA.AX             RSI   68.8       1.046            57.88              -11.03           TRADE     36.2      0.7203                26.1
    US     PG Bollinger Bands   63.6       0.914            47.88              -11.51           TRADE     22.2    

In [14]:
# ── Today's trade list (TRADE verdict, S/A/B only) ────────────────────────────
trade_list = df_results[
    (df_results["Today's Verdict"] == "TRADE") &
    (df_results["Tier"].isin(["S", "A", "B"]))
].sort_values("Score", ascending=False)

cols2 = ["Market", "Asset", "Best Strategy", "Tier", "Score",
         "Current Trend", "Kelly %", "Vol Scalar", "Recommended Size %"]
print(f"Today's Trade List ({len(trade_list)} assets):\n")
print(trade_list[cols2].to_string(index=False))

Today's Trade List (36 assets):

Market  Asset   Best Strategy Tier  Score Current Trend  Kelly %  Vol Scalar  Recommended Size %
   JPX 8750.T            MACD    S   75.8          Bull     14.4      0.5180                 7.5
   JPX 8411.T            MACD    S   74.7          Bull     13.1      0.3715                 4.9
   JPX 8306.T            MACD    S   74.5          Bull     16.4      0.5693                 9.3
    US   TSLA            MACD    B   69.4          Bull     18.2      0.3800                 6.9
   JPX 8002.T    MA Crossover    B   69.4          Bull     12.0      0.4241                 5.1
   ASX CBA.AX             RSI    S   68.8      Sideways     36.2      0.7203                26.1
   JPX 9434.T Bollinger Bands    A   68.1      Sideways     39.6      0.7624                30.2
    US    CAT            MACD    A   68.1          Bull     11.4      0.3415                 3.9
   JPX 7011.T            MACD    A   67.2          Bull     12.4      0.4063                 5

In [15]:
# ── Strategy distribution ─────────────────────────────────────────────────────
print("Strategy wins by market:\n")
dist = df_results.groupby(["Market", "Best Strategy"]).size().unstack(fill_value=0)
dist["Total"] = dist.sum(axis=1)
print(dist)

print("\nTop strategies overall:")
print(df_results["Best Strategy"].value_counts())

Strategy wins by market:

Best Strategy  Bollinger Bands  MA Crossover  MACD  Mean Reversion  RSI  Total
Market                                                                        
ASX                          4            11     6               3    5     29
JPX                         10            12     9               4   13     48
US                           8            11    12               3    6     40

Top strategies overall:
Best Strategy
MA Crossover       34
MACD               27
RSI                24
Bollinger Bands    22
Mean Reversion     10
Name: count, dtype: int64


In [ ]:
# ── Phase 5 sizing deep dive (S/A/B tier only) ───────────────────────────────
sized = df_results[
    (df_results["Kelly %"].notna()) & (df_results["Tier"] != "Skip")
].sort_values("Recommended Size %", ascending=False)

cols3 = ["Market", "Asset", "Tier", "N Trades", "Trade Win Rate %",
         "Avg Win %", "Avg Loss %", "Kelly %", "Vol Scalar", "Recommended Size %",
         "Today's Verdict"]
print(f"Position sizing breakdown — top 20 by recommended size (S/A/B tier only):\n")
print(sized[cols3].head(20).to_string(index=False))

---
## Part 8 — Save to Excel

In [17]:
# ── Save results to Excel ─────────────────────────────────────────────────────
fname = f"Goofy_Phase5_notebook_{today}.xlsx"
path  = os.path.join(OUTPUT_DIR, fname)

with pd.ExcelWriter(path, engine="openpyxl") as writer:
    # All results
    df_results.sort_values("Score", ascending=False).to_excel(
        writer, sheet_name="All Results", index=False)

    # Today's trade list
    trade_list.to_excel(writer, sheet_name="Today's Trade List", index=False)

    # S & A tier
    sa = df_results[df_results["Tier"].isin(["S", "A"])].sort_values("Score", ascending=False)
    sa.to_excel(writer, sheet_name="S and A Tier", index=False)

    # Position sizing
    sized.to_excel(writer, sheet_name="Position Sizing", index=False)

print(f"Saved: {path}")
import subprocess
subprocess.Popen(["open", path])

Saved: /Users/hiro/quant-research/Claude project (me learning)/Quant python learning 1/screener_output/Goofy_Phase5_notebook_2026-05-11.xlsx


<Popen: returncode: None args: ['open', '/Users/hiro/quant-research/Claude p...>